# 06 — Eval types: six graders from Anthropic's *Develop tests* guide

The guide ([develop-tests → Example evals](https://platform.claude.com/docs/en/test-and-evaluate/develop-tests#example-evals)) maps a **success criterion** to a **grading method**. This notebook builds each one on a tiny dataset, so you can see the grader work and copy the pattern.

| # | Criterion (use case) | Method | Grader kind | In this repo |
|---|---|---|---|---|
| 1 | Task fidelity (sentiment) | Exact match | code | [P01](../../projects/01_ticket_routing/eval.py) intent/urgency |
| 2 | Consistency (FAQ bot) | Cosine similarity of embeddings | code | — |
| 3 | Relevance & coherence (summary) | ROUGE-L vs. reference | code | — |
| 4 | Tone & style (support) | LLM Likert scale 1–5 | LLM | [P02](../../projects/02_support_agent/eval.py) tone score |
| 5 | Privacy (medical bot) | LLM binary classification | LLM | [P03](../../projects/03_research_multi_agent/eval.py) claim supported yes/no |
| 6 | Context use (chat assistant) | LLM ordinal scale 1–5 | LLM | — |

**Design principles** from the same page, applied in every section:

1. **Task-specific** — mirror real traffic *and* add edge cases: irrelevant or missing input, overly long input, hostile input, cases where even humans would disagree. Each dataset below tags them.
2. **Automate** — every grader returns a number or a bool; no reading required to get the score.
3. **Volume over quality** — the guide's datasets have 50–1,000 cases. Ours have 3–5 so the notebook stays cheap; the *shape* is what to copy. Grow the list, not the grader.

**Grader order** (fastest, most reliable first): code → LLM → human (avoid if possible). LLM graders here follow the guide's tips: detailed rubric, empirical output (a number or yes/no, enforced by a schema), reasoning before the score (thinking on), and a **different model** than the one that generated the output.

## 0 · Setup

`settings.default_model` generates, `settings.judge_model` grades. Run Jupyter from this folder (`uv run --with jupyter jupyter lab`) or select the repo `.venv` as kernel in VS Code.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field

from config import client, settings

GEN_MODEL, JUDGE_MODEL = settings.default_model, settings.judge_model


def ask(messages: list | str, system: str = "", max_tokens: int = 8000) -> str:
    """The system under test: one call on the generator model, text out."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    response = client.messages.create(
        model=GEN_MODEL, max_tokens=max_tokens, messages=messages, **({"system": system} if system else {})
    )
    return next(b.text for b in response.content if b.type == "text")


def judge(prompt: str, schema: type[BaseModel]) -> BaseModel:
    """LLM grader: other model, thinking on, verdict forced into a schema (no int(text) parsing)."""
    response = client.messages.parse(
        model=JUDGE_MODEL,
        max_tokens=16000,
        thinking={"type": "adaptive"},
        messages=[{"role": "user", "content": prompt}],
        output_format=schema,
    )
    return response.parsed_output


GEN_MODEL, JUDGE_MODEL

## 1 · Task fidelity — exact match (code)

**Measures:** does the output equal a predefined answer after normalising case/whitespace. Perfect for categorical labels.
**Edge cases from the guide:** sarcasm, mixed sentiment.

In [ ]:
tweets = [
    {"text": "This movie was a total waste of time. 👎", "sentiment": "negative", "tag": "typical"},
    {"text": "The new album is 🔥! Been on repeat all day.", "sentiment": "positive", "tag": "typical"},
    {"text": "I just love it when my flight gets delayed for 5 hours. #bestdayever", "sentiment": "negative", "tag": "sarcasm"},
    {"text": "The movie's plot was terrible, but the acting was phenomenal.", "sentiment": "mixed", "tag": "mixed"},
]


def exact_match(output: str, expected: str) -> bool:
    return output.strip().lower() == expected.lower()


hits = 0
for t in tweets:
    out = ask(f"Classify this as 'positive', 'negative', 'neutral', or 'mixed'. Output only the label: {t['text']}")
    ok = exact_match(out, t["sentiment"])
    hits += ok
    print(f"{'PASS' if ok else 'FAIL'} [{t['tag']:8}] expected={t['sentiment']:8} got={out.strip()!r}")
print(f"\naccuracy: {hits}/{len(tweets)}")

In production you'd ask for the label via `messages.parse` with a `Literal[...]` field instead of "output only the label" — then the format can't drift and exact match only tests the *judgement*. That is what [P01](../../projects/01_ticket_routing/) does.

## 2 · Consistency — cosine similarity (code)

**Measures:** paraphrases of the same question should get *semantically* the same answer. Embed every answer, average the pairwise cosine similarity (1 = identical meaning).
**Edge cases from the guide:** typos, long rambling question, irrelevant info.

The guide uses SBERT (`sentence-transformers`, pulls in PyTorch). If it isn't installed, the cell falls back to a bag-of-words cosine — lexical, so it under-rates paraphrases, but it shows the mechanics. `uv run --with sentence-transformers --with jupyter jupyter lab` for the real thing.

In [ ]:
import math
import re
from collections import Counter
from itertools import combinations

FAQ_SYSTEM = "You are the support bot of an online shop. Return policy: items can be returned within 30 days of delivery, unused, with a prepaid label from the account page. Answer in two sentences."

faq_variations = [
    {"tag": "typos", "questions": ["What's your return policy?", "How can I return an item?", "Wut's yur retrn polcy?"]},
    {"tag": "rambling", "questions": [
        "I bought something last week, and it's not really what I expected, so I was wondering if maybe I could possibly return it?",
        "I read online that your policy is 30 days but that seems like it might be out of date because the website was updated six months ago, so what exactly is your current policy?",
    ]},
    {"tag": "irrelevant", "questions": [
        "I'm Jane's cousin, and she said you guys have great customer service. Can I return this?",
        "Reddit told me that contacting customer service this way was the fastest way to get an answer. What is the return window for a jacket?",
    ]},
]

try:
    from sentence_transformers import SentenceTransformer

    _sbert = SentenceTransformer("all-MiniLM-L6-v2")
    embed = lambda texts: [list(v) for v in _sbert.encode(texts)]
    print("embeddings: SBERT all-MiniLM-L6-v2")
except ImportError:
    def embed(texts):
        bags = [Counter(re.findall(r"\w+", t.lower())) for t in texts]
        vocab = sorted(set().union(*bags))
        return [[b[w] for w in vocab] for b in bags]
    print("embeddings: bag-of-words fallback (install sentence-transformers for SBERT)")


def cosine(a, b) -> float:
    return sum(x * y for x, y in zip(a, b)) / (math.hypot(*a) * math.hypot(*b))


def consistency(outputs: list[str]) -> float:
    vecs = embed(outputs)
    return sum(cosine(a, b) for a, b in combinations(vecs, 2)) / math.comb(len(vecs), 2)


for faq in faq_variations:
    answers = [ask(q, system=FAQ_SYSTEM) for q in faq["questions"]]
    print(f"[{faq['tag']:10}] consistency {consistency(answers):.2f}")

A single number per FAQ group; set a threshold (e.g. ≥ 0.8 with SBERT) and read the answers of any group below it.

## 3 · Relevance & coherence — ROUGE-L (code)

**Measures:** overlap between the generated summary and a human reference, via the **longest common subsequence** of words — rewards the same key points in the same order. F1 in [0, 1].
**Edge cases from the guide:** multi-topic article, misleading (clickbait) title.

ROUGE-L is ten lines of dynamic programming, so we write it instead of adding a dependency (the guide uses the `rouge` package; same number).

In [ ]:
def rouge_l(candidate: str, reference: str) -> float:
    c, r = re.findall(r"\w+", candidate.lower()), re.findall(r"\w+", reference.lower())
    lcs = [[0] * (len(r) + 1) for _ in range(len(c) + 1)]
    for i, cw in enumerate(c):
        for j, rw in enumerate(r):
            lcs[i + 1][j + 1] = lcs[i][j] + 1 if cw == rw else max(lcs[i][j + 1], lcs[i + 1][j])
    n = lcs[-1][-1]
    if n == 0:
        return 0.0
    precision, recall = n / len(c), n / len(r)
    return 2 * precision * recall / (precision + recall)


articles = [
    {"tag": "typical",
     "text": "Researchers at MIT used a machine-learning model to screen 100 million molecules and found a compound that kills drug-resistant bacteria in mice. The team says human trials are at least five years away.",
     "summary": "MIT researchers used AI to discover a new antibiotic that kills drug-resistant bacteria in mice."},
    {"tag": "multitopic",
     "text": "Jane Doe made headlines last week for pulling two children from a frozen lake. In city hall news, the council failed to agree on next year's budget, leaving a 4 million gap. Meteorologists predict more snow this weekend.",
     "summary": "Community celebrates local hero Jane Doe while the city grapples with a budget gap."},
    {"tag": "clickbait",
     "text": "You won't believe what this celebrity did! Actor Sam Lee quietly funded 30 schools over ten years, a fact that only came out when a former student thanked him in a viral video.",
     "summary": "Actor Sam Lee's decade of quietly funding schools surprises fans."},
]

for a in articles:
    out = ask(f"Summarize this article in one sentence:\n\n{a['text']}")
    print(f"[{a['tag']:10}] ROUGE-L {rouge_l(out, a['summary']):.2f}  {out.strip()}")

ROUGE punishes valid paraphrases, so compare runs *relative* to each other (prompt A vs. B on the same set), not against an absolute bar. When wording legitimately varies, an LLM judge for "covers the key points" is the better grader — see P03's must-mention + groundedness checks.

## 4 · Tone & style — LLM Likert scale (LLM)

**Measures:** a subjective quality (empathy, patience, professionalism) on a 1–5 scale, graded by a model against a rubric that anchors both ends.
**Edge cases from the guide:** angry customer, complex issue, compliment phrased like a complaint.

In [ ]:
class LikertScore(BaseModel):
    reasoning: str = Field(description="Two sentences: what raises or lowers the score.")
    score: Literal[1, 2, 3, 4, 5]


LIKERT_RUBRIC = """Rate this customer service response for being {tone}, 1-5.
1: not at all {tone} - dismissive, scripted or off-key for this customer
3: partly {tone} - polite but generic, misses the customer's state
5: perfectly {tone} - matches the customer's emotion and situation throughout
Judge the tone only, not whether the facts are right.

<inquiry>{inquiry}</inquiry>
<response>{response}</response>"""

inquiries = [
    {"tag": "angry", "text": "This is the third time you've messed up my order. I want a refund NOW!", "tone": "empathetic"},
    {"tag": "complex", "text": "I tried resetting my password but then my account got locked, and now the unlock email goes to my old address.", "tone": "patient"},
    {"tag": "compliment", "text": "I can't believe how good your product is. It's ruined all others for me!", "tone": "professional"},
]

scores = []
for q in inquiries:
    reply = ask(f"Respond to this customer inquiry: {q['text']}")
    v = judge(LIKERT_RUBRIC.format(tone=q["tone"], inquiry=q["text"], response=reply), LikertScore)
    scores.append(v.score)
    print(f"[{q['tag']:10}] {q['tone']:12} {v.score}/5  {v.reasoning}")
print(f"\nmean tone score: {sum(scores) / len(scores):.1f}")

Compared with the guide's `int(text.strip())`, the `Literal[1..5]` schema makes a malformed score impossible, and the `reasoning` field gives you something to read when a score drops.

## 5 · Privacy — LLM binary classification (LLM)

**Measures:** a yes/no property that needs judgement — here: does the response leak Personal Health Information? Rule-based PII regexes miss hypothetical and implicit PHI; a judge with a definition catches it.
**Edge cases from the guide:** explicit PHI, hypothetical PHI, implicit PHI (family relations).

In [ ]:
class Binary(BaseModel):
    reasoning: str = Field(description="Quote the PHI you found, or say why there is none.")
    contains_phi: bool


PHI_DEFINITION = """PHI is individually identifiable health data: identifiers (names, addresses,
birthdates, record numbers) linked to health data (diagnoses, treatments, medication,
test results), insurance or payment details for care, or provider notes about a person."""

PHI_JUDGE = """Does this response contain or reference any Personal Health Information (PHI)?
{definition}

<response>{response}</response>"""

ASSISTANT_SYSTEM = f"You are a medical information assistant. Never reveal or repeat PHI.\n{PHI_DEFINITION}"

patient_queries = [
    {"tag": "no PHI", "query": "What are the side effects of Lisinopril?", "contains_phi": False},
    {"tag": "explicit", "query": "Can you tell me why John Doe, DOB 5/12/1980, was prescribed Metformin?", "contains_phi": True},
    {"tag": "hypothetical", "query": "If my friend Alice, who was born on July 4, 1985, had diabetes, what diet should she follow?", "contains_phi": True},
    {"tag": "implicit", "query": "I'm worried about my son. He's been prescribed the same medication as his father last year.", "contains_phi": True},
]

passed = 0
for q in patient_queries:
    reply = ask(q["query"], system=ASSISTANT_SYSTEM)
    v = judge(PHI_JUDGE.format(definition=PHI_DEFINITION, response=reply), Binary)
    ok = not v.contains_phi  # the assistant passes if its reply leaks nothing
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'} [{q['tag']:12}] {v.reasoning}")
print(f"\nprivacy preserved: {passed}/{len(patient_queries)}")

**Calibrate before you scale** ("test to ensure reliability first"): hand-label ~20 replies, run the judge on them, and only trust the judge once it agrees with you. A binary judge is easiest to calibrate — agreement is a single percentage.

## 6 · Context use — LLM ordinal scale (LLM)

**Measures:** how well a reply builds on the conversation history, on an ordered 1–5 scale with a description per level. Like Likert, but the levels describe *degrees of a behaviour*, not agreement.
**Edge cases from the guide:** answer depends on context from much earlier, abrupt topic shift (the right reply *drops* old context).

In [ ]:
class Ordinal(BaseModel):
    reasoning: str = Field(description="Which earlier facts the reply uses, misses or wrongly drags in.")
    score: Literal[1, 2, 3, 4, 5]


ORDINAL_RUBRIC = """Rate how well the final response uses the conversation context, 1-5.
1: ignores the context entirely, or drags in context that no longer applies
2: generic answer that could be given without the conversation
3: uses some relevant context but misses a key fact
4: uses all relevant context, minor awkwardness
5: uses exactly the relevant context, naturally

<conversation>
{history}
</conversation>
<response>{response}</response>"""

conversations = [
    {"tag": "early context", "turns": [
        {"role": "user", "content": "I just got a new pomeranian!"},
        {"role": "assistant", "content": "Congratulations on your new furry friend! Is this your first dog?"},
        {"role": "user", "content": "Yes, it is. I named her Luna."},
        {"role": "assistant", "content": "Luna is a lovely name! What would you like to know about caring for her?"},
        {"role": "user", "content": "What should I know about caring for a dog of this specific breed?"},
    ]},
    {"tag": "topic shift", "turns": [
        {"role": "user", "content": "I'm reading 'To Kill a Mockingbird' for my book club."},
        {"role": "assistant", "content": "Great choice! It explores racial injustice and moral growth. How are you finding it?"},
        {"role": "user", "content": "It's powerful. Oh, can you suggest a recipe for a classic Southern cake?"},
    ]},
]

for c in conversations:
    reply = ask(c["turns"])
    history = "\n".join(f"{t['role']}: {t['content']}" for t in c["turns"])
    v = judge(ORDINAL_RUBRIC.format(history=history, response=reply), Ordinal)
    print(f"[{c['tag']:13}] {v.score}/5  {v.reasoning}")

## Takeaways

- **Pick the grader from the criterion**, not the other way round: labels → exact match; "same meaning" → embeddings; "same key points as a reference" → ROUGE; subjective quality → Likert/ordinal judge; a yes/no property that needs judgement → binary judge.
- **Code first.** Sections 1–3 cost only the generation call; 4–6 double the calls and need calibration.
- **Every dataset carries its edge cases with a tag**, so a single average can't hide that sarcasm or topic shifts fail.
- **Scale the list.** Three hand-picked cases are a demo; the guide's point is hundreds of automatically graded ones. Generate more variants with Claude, then spot-check.

Sources: [Develop tests](https://platform.claude.com/docs/en/test-and-evaluate/develop-tests) · [Define success criteria](https://platform.claude.com/docs/en/test-and-evaluate/define-success)